In [2]:

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

def load_split(name):
    path = ARTIFACTS_DIR / f"{name}_processed.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run the save cell in Notebook 2 first."
        )
    return pd.read_parquet(path)

train_df = load_split("train")
val_df = load_split("validation")
test_df = load_split("test")

X_train_scaled = train_df.drop(columns="Class")
y_train = train_df["Class"]

X_val_scaled = val_df.drop(columns="Class")
y_val = val_df["Class"]

X_test_scaled = test_df.drop(columns="Class")
y_test = test_df["Class"]

print("Train:", X_train_scaled.shape, "| Fraud:", int(y_train.sum()))
print("Validation:", X_val_scaled.shape, "| Fraud:", int(y_val.sum()))
print("Test:", X_test_scaled.shape, "| Fraud:", int(y_test.sum()))


Train: (198608, 30) | Fraud: 331
Validation: (42559, 30) | Fraud: 71
Test: (42559, 30) | Fraud: 71


In [3]:

# Model 1: Logistic Regression
logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
)

logistic_model.fit(X_train_scaled, y_train)

# Model 2: Decision Tree
tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    max_depth=5,
    min_samples_leaf=10,
    random_state=RANDOM_STATE,
)

tree_model.fit(X_train_scaled, y_train)

print("Both baseline models trained successfully.")
print(f"Logistic Regression iterations: {logistic_model.n_iter_[0]}")
print(f"Decision Tree depth: {tree_model.get_depth()}")
print(f"Decision Tree leaves: {tree_model.get_n_leaves()}")
print("Class weighting: balanced")
print("Resampling: none")


Both baseline models trained successfully.
Logistic Regression iterations: 44
Decision Tree depth: 5
Decision Tree leaves: 25
Class weighting: balanced
Resampling: none


In [4]:

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
}

validation_results = []
validation_confusion_matrices = {}

for model_name, model in models.items():
    # Predict probabilities and apply the default 0.5 threshold
    y_val_proba = model.predict_proba(X_val_scaled)[:, 1]
    y_val_pred = (y_val_proba >= 0.5).astype(int)

    validation_results.append({
        "Model": model_name,
        "Precision": precision_score(y_val, y_val_pred, zero_division=0),
        "Recall": recall_score(y_val, y_val_pred, zero_division=0),
        "F1": f1_score(y_val, y_val_pred, zero_division=0),
        "Average Precision": average_precision_score(y_val, y_val_proba),
        "Predicted Fraud": int(y_val_pred.sum()),
    })

    validation_confusion_matrices[model_name] = confusion_matrix(
        y_val, y_val_pred, labels=[0, 1]
    )

results_df = pd.DataFrame(validation_results).set_index("Model")
display(results_df.round(4))

for model_name, matrix in validation_confusion_matrices.items():
    print(f"\n{model_name} — Confusion Matrix")
    print("Rows = actual [legitimate, fraud]")
    print("Columns = predicted [legitimate, fraud]")
    print(matrix)


,Precision,Recall,F1,Average Precision,Predicted Fraud
Model,,,,,
Logistic Regression,0.0613,0.9155,0.1149,0.7196,1060
Decision Tree,0.0731,0.9014,0.1353,0.4744,875



Logistic Regression — Confusion Matrix
Rows = actual [legitimate, fraud]
Columns = predicted [legitimate, fraud]
[[41493   995]
 [    6    65]]

Decision Tree — Confusion Matrix
Rows = actual [legitimate, fraud]
Columns = predicted [legitimate, fraud]
[[41677   811]
 [    7    64]]


In [5]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Reuse the already-trained model and validation data
logistic_val_proba = logistic_model.predict_proba(X_val_scaled)[:, 1]

threshold_results = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    predictions = (logistic_val_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "Predicted Fraud": int(predictions.sum()),
        "False Positives": int(
            ((predictions == 1) & (y_val.to_numpy() == 0)).sum()
        ),
        "False Negatives": int(
            ((predictions == 0) & (y_val.to_numpy() == 1)).sum()
        ),
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


,Threshold,Precision,Recall,F1,Predicted Fraud,False Positives,False Negatives
0,0.1,0.0088,0.9577,0.0174,7737,7669,3
1,0.2,0.0178,0.9296,0.0349,3711,3645,5
2,0.3,0.0288,0.9155,0.0558,2257,2192,6
3,0.4,0.0429,0.9155,0.0820,1514,1449,6
4,0.5,0.0613,0.9155,0.1149,1060,995,6
5,0.6,0.0814,0.9014,0.1494,786,722,7
6,0.7,0.1087,0.9014,0.1939,589,525,7
7,0.8,0.1391,0.9014,0.2411,460,396,7
8,0.9,0.1862,0.8732,0.3069,333,271,9


In [6]:

from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    predictions = (logistic_val_proba >= threshold).astype(int)

    tp = int(((predictions == 1) & (y_val.to_numpy() == 1)).sum())
    fp = int(((predictions == 1) & (y_val.to_numpy() == 0)).sum())
    fn = int(((predictions == 0) & (y_val.to_numpy() == 1)).sum())

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1": f1_score(y_val, predictions, zero_division=0),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Predicted Fraud": int(predictions.sum()),
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.style
    .format({
        "Threshold": "{:.1f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
    })
    .hide(axis="index")
)

assert (threshold_df["TP"] + threshold_df["FN"] == int(y_val.sum())).all()
assert (threshold_df["TP"] + threshold_df["FP"] == threshold_df["Predicted Fraud"]).all()


Threshold,Precision,Recall,F1,TP,FP,FN,Predicted Fraud
0.1,0.0088,0.9577,0.0174,68,7669,3,7737
0.2,0.0178,0.9296,0.0349,66,3645,5,3711
0.3,0.0288,0.9155,0.0558,65,2192,6,2257
0.4,0.0429,0.9155,0.0820,65,1449,6,1514
0.5,0.0613,0.9155,0.1149,65,995,6,1060
0.6,0.0814,0.9014,0.1494,64,722,7,786
0.7,0.1087,0.9014,0.1939,64,525,7,589
0.8,0.1391,0.9014,0.2411,64,396,7,460
0.9,0.1862,0.8732,0.3069,62,271,9,333


In [7]:

# Provisional business costs (USD per transaction)
COST_FN = 1000
COST_FP = 25
COST_TP = 0
COST_TN = 0

threshold_cost_results = []

for threshold in np.arange(0.01, 1.00, 0.01):
    predictions = (logistic_val_proba >= threshold).astype(int)
    actual = y_val.to_numpy()

    tp = int(((predictions == 1) & (actual == 1)).sum())
    fp = int(((predictions == 1) & (actual == 0)).sum())
    fn = int(((predictions == 0) & (actual == 1)).sum())
    tn = int(((predictions == 0) & (actual == 0)).sum())

    total_cost = (
        COST_FN * fn
        + COST_FP * fp
        + COST_TP * tp
        + COST_TN * tn
    )

    threshold_cost_results.append({
        "Threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": tp / (tp + fp) if tp + fp else 0,
        "Recall": tp / (tp + fn) if tp + fn else 0,
        "Total Cost ($)": total_cost,
        "Cost per Transaction ($)": total_cost / len(actual),
    })

cost_df = pd.DataFrame(threshold_cost_results)

# Rank thresholds by estimated validation cost
cost_df = cost_df.sort_values(
    ["Total Cost ($)", "Threshold"],
    ascending=[True, True],
).reset_index(drop=True)

print("Lowest-cost validation thresholds:")
display(cost_df.head(10).round(4))

best_threshold = float(cost_df.iloc[0]["Threshold"])
best_cost = float(cost_df.iloc[0]["Total Cost ($)"])

print(f"Selected threshold: {best_threshold:.2f}")
print(f"Validation total cost: ${best_cost:,.2f}")
print(f"Validation cost per transaction: ${best_cost / len(y_val):.4f}")


Lowest-cost validation thresholds:


,Threshold,TP,FP,FN,TN,Precision,Recall,Total Cost ($),Cost per Transaction ($)
0,0.98,62,71,9,42417,0.4662,0.8732,10775,0.2532
1,0.99,61,47,10,42441,0.5648,0.8592,11175,0.2626
2,0.97,62,99,9,42389,0.3851,0.8732,11475,0.2696
3,0.96,62,120,9,42368,0.3407,0.8732,12000,0.2820
4,0.95,62,154,9,42334,0.2870,0.8732,12850,0.3019
5,0.94,62,185,9,42303,0.2510,0.8732,13625,0.3201
6,0.93,62,216,9,42272,0.2230,0.8732,14400,0.3384
7,0.92,62,233,9,42255,0.2102,0.8732,14825,0.3483
8,0.91,62,256,9,42232,0.1950,0.8732,15400,0.3619
9,0.84,64,350,7,42138,0.1546,0.9014,15750,0.3701


Selected threshold: 0.98
Validation total cost: $10,775.00
Validation cost per transaction: $0.2532


In [8]:

default_row = cost_df.loc[
    np.isclose(cost_df["Threshold"], 0.50)
].iloc[0]

best_row = cost_df.iloc[0]

comparison = pd.DataFrame([
    {
        "Strategy": "Default threshold (0.50)",
        "Threshold": default_row["Threshold"],
        "False Positives": int(default_row["FP"]),
        "False Negatives": int(default_row["FN"]),
        "Recall": default_row["Recall"],
        "Total Cost ($)": default_row["Total Cost ($)"],
    },
    {
        "Strategy": "Minimum validation cost",
        "Threshold": best_row["Threshold"],
        "False Positives": int(best_row["FP"]),
        "False Negatives": int(best_row["FN"]),
        "Recall": best_row["Recall"],
        "Total Cost ($)": best_row["Total Cost ($)"],
    },
])

display(comparison.round(4))

savings = default_row["Total Cost ($)"] - best_row["Total Cost ($)"]
print(f"Estimated validation savings: ${savings:,.2f}")


,Strategy,Threshold,False Positives,False Negatives,Recall,Total Cost ($)
0,Default threshold (0.50),0.50,995,6,0.9155,30875.0
1,Minimum validation cost,0.98,71,9,0.8732,10775.0


Estimated validation savings: $20,100.00


In [9]:

# Inspect a finer grid near the current optimum.
# Validation data only; no test predictions are used.

fine_thresholds = np.arange(0.950, 1.000, 0.001)
fine_results = []

actual = y_val.to_numpy()

for threshold in fine_thresholds:
    predictions = (logistic_val_proba >= threshold).astype(int)

    tp = int(((predictions == 1) & (actual == 1)).sum())
    fp = int(((predictions == 1) & (actual == 0)).sum())
    fn = int(((predictions == 0) & (actual == 1)).sum())

    fine_results.append({
        "Threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Recall": tp / (tp + fn) if tp + fn else 0,
        "Total Cost ($)": COST_FN * fn + COST_FP * fp,
    })

fine_cost_df = pd.DataFrame(fine_results)
fine_cost_df = fine_cost_df.sort_values(
    ["Total Cost ($)", "Threshold"],
    ascending=[True, True],
).reset_index(drop=True)

display(fine_cost_df.head(10).round(4))


,Threshold,TP,FP,FN,Recall,Total Cost ($)
0,0.982,62,68,9,0.8732,10700
1,0.980,62,71,9,0.8732,10775
2,0.981,62,71,9,0.8732,10775
3,0.979,62,73,9,0.8732,10825
4,0.978,62,74,9,0.8732,10850
5,0.977,62,76,9,0.8732,10900
6,0.976,62,80,9,0.8732,11000
7,0.975,62,82,9,0.8732,11050
8,0.993,61,42,10,0.8592,11050
9,0.992,61,44,10,0.8592,11100


In [10]:

# Verify the cost arithmetic and sort by threshold for readability.
fine_cost_df["Recalculated Cost ($)"] = (
    COST_FN * fine_cost_df["FN"]
    + COST_FP * fine_cost_df["FP"]
)

fine_cost_df["Cost Check"] = (
    fine_cost_df["Total Cost ($)"]
    == fine_cost_df["Recalculated Cost ($)"]
)

display(
    fine_cost_df.sort_values("Threshold")[
        [
            "Threshold", "TP", "FP", "FN", "Recall",
            "Recalculated Cost ($)", "Cost Check"
        ]
    ].round(4).head(20)
)

best_fine = fine_cost_df.sort_values(
    ["Recalculated Cost ($)", "Threshold"],
    ascending=[True, True],
).iloc[0]

print(f"Best fine-grid threshold: {best_fine['Threshold']:.3f}")
print(f"Minimum validation cost: ${best_fine['Recalculated Cost ($)']:,.2f}")


,Threshold,TP,FP,FN,Recall,Recalculated Cost ($),Cost Check
45,0.950,62,154,9,0.8732,12850,True
44,0.951,62,152,9,0.8732,12800,True
42,0.952,62,149,9,0.8732,12725,True
41,0.953,62,147,9,0.8732,12675,True
39,0.954,62,141,9,0.8732,12525,True
38,0.955,62,137,9,0.8732,12425,True
37,0.956,62,135,9,0.8732,12375,True
36,0.957,62,133,9,0.8732,12325,True
35,0.958,62,128,9,0.8732,12200,True
34,0.959,62,125,9,0.8732,12125,True


Best fine-grid threshold: 0.982
Minimum validation cost: $10,700.00


In [11]:

# Sensitivity analysis: how the assumed cost of missed fraud affects
# the optimal Logistic Regression threshold.

from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

COST_FP = 25
FN_COST_SCENARIOS = [250, 500, 1000]

sensitivity_results = []

thresholds = np.arange(0.001, 1.000, 0.001)

for cost_fn in FN_COST_SCENARIOS:
    scenario_rows = []

    for threshold in thresholds:
        predictions = (logistic_val_proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_val, predictions, labels=[0, 1]
        ).ravel()

        total_cost = cost_fn * fn + COST_FP * fp

        scenario_rows.append({
            "Threshold": threshold,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Recall": tp / (tp + fn) if (tp + fn) else 0,
            "Total Cost ($)": total_cost,
        })

    scenario_df = pd.DataFrame(scenario_rows)
    best = scenario_df.sort_values(
        ["Total Cost ($)", "Threshold"],
        ascending=[True, False]
    ).iloc[0]

    sensitivity_results.append({
        "Missed Fraud Cost ($)": cost_fn,
        "False Positive Cost ($)": COST_FP,
        "Best Threshold": best["Threshold"],
        "TP": int(best["TP"]),
        "FP": int(best["FP"]),
        "FN": int(best["FN"]),
        "Recall": best["Recall"],
        "Minimum Validation Cost ($)": int(best["Total Cost ($)"]),
    })

display(pd.DataFrame(sensitivity_results).round(4))


,Missed Fraud Cost ($),False Positive Cost ($),Best Threshold,TP,FP,FN,Recall,Minimum Validation Cost ($)
0,250,25,0.993,61,42,10,0.8592,3550
1,500,25,0.993,61,42,10,0.8592,6050
2,1000,25,0.982,62,68,9,0.8732,10700


In [12]:

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

COST_FN = 1000
COST_FP = 25
thresholds = np.arange(0.001, 1.000, 0.001)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
}

comparison_results = []

for model_name, model in models.items():
    val_probabilities = model.predict_proba(X_val_scaled)[:, 1]
    rows = []

    for threshold in thresholds:
        predictions = (val_probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_val, predictions, labels=[0, 1]
        ).ravel()

        rows.append({
            "Threshold": threshold,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Recall": tp / (tp + fn),
            "Total Cost ($)": COST_FN * fn + COST_FP * fp,
        })

    results = pd.DataFrame(rows)
    best = results.sort_values(
        ["Total Cost ($)", "Threshold"],
        ascending=[True, False]
    ).iloc[0]

    comparison_results.append({
        "Model": model_name,
        "Best Threshold": best["Threshold"],
        "TP": int(best["TP"]),
        "FP": int(best["FP"]),
        "FN": int(best["FN"]),
        "Recall": best["Recall"],
        "Minimum Validation Cost ($)": int(best["Total Cost ($)"]),
    })

display(pd.DataFrame(comparison_results).round(4))


,Model,Best Threshold,TP,FP,FN,Recall,Minimum Validation Cost ($)
0,Logistic Regression,0.982,62,68,9,0.8732,10700
1,Decision Tree,0.998,60,49,11,0.8451,12225


In [13]:

import joblib

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

models_to_save = {
    "logistic_regression.joblib": logistic_model,
    "decision_tree.joblib": tree_model,
}

for filename, model in models_to_save.items():
    output_path = ARTIFACTS_DIR / filename
    joblib.dump(model, output_path)
    print(f"Saved: {output_path}")

print("\nModel saving complete.")


Saved: c:\Users\Administrator\Self Study\fraud_detection\fraud-detection\fraud-detection\artifacts\logistic_regression.joblib
Saved: c:\Users\Administrator\Self Study\fraud_detection\fraud-detection\fraud-detection\artifacts\decision_tree.joblib

Model saving complete.
